In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
import scipy
from signed_inference import *


In [ ]:
from graph_tool.all import *


In [ ]:
x=12

In [ ]:
tempstrmx="inf_mtx_dec2long_"+str(x)+".moresteps_real.npy"
mx=np.load(os.getcwd() + str("/diff_days/"+'inf_mtx_04.npy'))
point_est=sum(mx)/len(mx)
plt.imshow(point_est,cmap=plt.cm.bwr)

In [ ]:
g= Graph(scipy.sparse.lil_matrix(point_est))

In [ ]:
print(g)

In [ ]:
"""
mx=np.load('inf_mtx.npy')
point_est=sum(mx)/len(mx)
plt.imshow(point_est,cmap=plt.cm.bwr,vmin=-1, vmax=1)
"""

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from datetime import datetime, timedelta

df_id=pd.read_csv('ids_with_index.txt',header=None)
df_id.head()
ids=pd.DataFrame.to_numpy(df_id)

df = pd.read_csv('High-School_data_2013.csv', delimiter=' ')
df.head()


x=pd.DataFrame.to_numpy(df)
def lookup(key, dict):
    return dict[key]
id_dict={}
for i in ids:
    id_dict[i[1]]=i[0]
print(id_dict)
for i in range(len(x)):
    #print(x[i][1])
    #print(id_dict[x[i][1]])
    #id_dict[x[i][1]]
    x[i][2]=id_dict[x[i][2]]
    x[i][1]=id_dict[x[i][1]]
print(x)
classes=set(df['class1'])
print(classes)
classes_dict={}
curr_key=0
for i in classes:
    classes_dict[i]=curr_key
    curr_key+=1
classes_dict
memberships=np.zeros(len(point_est))
for i in x:
    memberships[i[1]]=classes_dict[i[3]]
    memberships[i[2]]=classes_dict[i[4]]
print(memberships)

ordering=np.argsort(memberships)
classes_reverse = {y: x for x, y in classes_dict.items()}



In [ ]:
def plot_histograms(inferences,parts,savetitle=None,plt_title=None):
    entropy_mtx=np.zeros((len(inferences[0]),len(inferences[0])))
    
    mplus,mneg,mzer=[],[],[]
    for i in inferences:
        mplus.append((i>0).astype(int))
        mneg.append((i<0).astype(int))
        mzer.append((i==0).astype(int))
    
    mplus_agg=sum(mplus)
    mneg_agg=sum(mneg)
    mzer_agg=sum(mzer)
    for n in range(len(mplus_agg)):
        for m in range(n):
            entropy_mtx[n][m]=scipy.stats.entropy([mplus_agg[n][m],mneg_agg[n][m],mzer_agg[n][m]])
            entropy_mtx[m][n]=entropy_mtx[n][m]
    return entropy_mtx
    temp=np.zeros((len(parts[0]),len(parts[0])))
    
    for i in range(len(temp)):
        for j in range(i):
            found=False
            curr=0
            while found==False and curr <len(parts):
                if parts[curr][i]==parts[curr][j]:
                    found=True
                    temp[i][j]=1
                    temp[j][i]=1
                curr+=1
    tgthr=[]
    sprtly=[]
    for i in range(len(entropy_mtx)):
        for j in range(i):
            if temp[i][j]==1:
                tgthr.append(entropy_mtx[i][j])
            else:
                sprtly.append(entropy_mtx[i][j])
    """
    plt.hist([tgthr, sprtly], color=['Blue', 'Red'], label=['Edge inside a partition', 'Edge between partitions '])
    plt.xlabel('Entropy')
    plt.title(plt_title)
    """
    tgthr=np.transpose(tgthr)
    sprtly=np.transpose(sprtly)
    new_vec1=[]
    new_vec2=[]
    for i in range(len(tgthr)):
        new_vec1.append(tgthr[i])
        new_vec2.append("Inside partition")
    for i in range(len(sprtly)):
        new_vec1.append(sprtly[i])
        new_vec2.append("Between partitions")

    df=pd.DataFrame([new_vec1,new_vec2])
    """
    plt.ylabel('Entropy')
    plt.legend()
    plt.savefig('asdf')
    if savetitle!=None:
        plt.savefig(str(savetitle))
    plt.show()
    """
    #sns.histplot([tgthr,sprtly],kde=True,cumulative=False,cbar=True)
    df=df.transpose()
    df=df.rename(columns={0:'Entropy',1:'Edges'})
    sns.histplot(data=df, x='Entropy',palette=['#154cb0', '#de1212'], hue='Edges',kde=True,multiple="layer")
    plt.title(plt_title)
    plt.savefig(savetitle)
    plt.show()
    return df
partitions=np.array(np.zeros((len(point_est),1)))
d=plot_histograms(mx,partitions,'cico.png')

In [ ]:

separators=[0]
separators_lab=[classes_reverse[memberships[ordering[0]]]]
for i in range(len(ordering)-1):
    if memberships[ordering[i]]!=memberships[ordering[i+1]]:
        separators.append(i)
        separators_lab.append(classes_reverse[memberships[ordering[i+1]]])
print(separators)
print(classes_reverse)
print(separators_lab)

#separators=[]
for i in classes:
    print(i)
    for j in ordering:
        print(j)

In [ ]:
print(separators)

In [ ]:
separarators_plot=np.copy(separators)
separarators_plot=list(separarators_plot)
separarators_plot.append(327)
temp=[]
for i in range(len(separarators_plot)-1):
    temp.append((separarators_plot[i]+separarators_plot[i+1])/2)
print(len(temp))
separarators_plot=temp
len(separarators_plot)


In [ ]:
plt.imshow(d[ordering,:][:,ordering],cmap=plt.cm.gist_yarg)
for i in separators:
    print(i)
    plt.axvline(i - 0.5, color='blue')
    plt.axhline(i - 0.5, color='blue')
plt.xticks(separarators_plot,separators_lab)
plt.xticks(rotation='vertical')
plt.yticks(separarators_plot,separators_lab)

plt.colorbar()
plt.savefig('high_school_entropy_dec04.svg',dpi=500)
plt.show()
plt.imshow(point_est[ordering,:][:,ordering],cmap=plt.cm.bwr,vmin=-1)
for i in separators:
    print(i)
    plt.axvline(i - 0.5, color='black')
    plt.axhline(i - 0.5, color='black')
plt.xticks(separarators_plot,separators_lab)
plt.xticks(rotation='vertical')
plt.yticks(separarators_plot,separators_lab)
plt.colorbar()
plt.savefig('high_school_sampleavrg_dec04.svg',dpi=500)

In [ ]:
print(classes)

In [ ]:
print(classes)

In [ ]:
np.argmax(ordering==3)

In [ ]:
ordering

In [ ]:
def get_masked_matrix(M ,node_mask):
    """Get masked adjacency matrices.
    
    Parameters
    ----------
    N : ndarray
        Negative adjacency matrix.
    P : ndarray
        Positive adjacency matrix.
    node_mask : ndarray
        Boolean array of nodes to mask.
    """
    return M[node_mask,:][:,node_mask]

In [ ]:
classes_dict

In [ ]:
mask1=(memberships==classes_dict['MP*1'])
mask2=(memberships==classes_dict['MP*2'])
mask=np.copy(mask1)
memb_psi_mp2=[]
for i in range(len(mask1)):
    if mask1[i]==True or mask2[i]==True:
        mask[i]=True
        memb_psi_mp2.append(memberships[i])

    else:
        mask[i]=False

In [ ]:
plt.xticks(separarators_plot,separators_lab)
plt.xticks(rotation='vertical')
plt.yticks(separarators_plot,separators_lab)

In [ ]:
psi_mp2_odering=np.argsort(memb_psi_mp2)
sep_psi_mp2=[0]
sep_psi_mp2_lab=[classes_reverse[memb_psi_mp2[psi_mp2_odering[0]]]]
for i in range(len(psi_mp2_odering)-1):
    if memb_psi_mp2[psi_mp2_odering[i]]!=memb_psi_mp2[psi_mp2_odering[i+1]]:
        sep_psi_mp2.append(i)
        sep_psi_mp2_lab.append(classes_reverse[memb_psi_mp2[psi_mp2_odering[i+1]]])
sep_psi_mp2_line=list(np.copy(sep_psi_mp2))
sep_psi_mp2_line.append(len(psi_mp2_odering)-1)
temp=[]
for i in range(len(sep_psi_mp2_line)-1):
    temp.append((sep_psi_mp2_line[i]+sep_psi_mp2_line[i+1])*0.5)
sep_psi_mp2_line=temp
print("line:",sep_psi_mp2_line)
print(sep_psi_mp2)
print(classes_reverse)
print(sep_psi_mp2_lab)
    

In [ ]:
plt.imshow(get_masked_matrix(point_est,mask)[psi_mp2_odering,:][:,psi_mp2_odering],cmap=plt.cm.bwr,vmin=-1)
plt.title(str('PSI*2 and MP*2 posterior sample average'))
for i in sep_psi_mp2:
    print(i)
    plt.axvline(i - 0.5, color='blue')
    plt.axhline(i - 0.5, color='blue')
plt.xticks(sep_psi_mp2_line,sep_psi_mp2_lab)
plt.xticks(rotation='vertical')
plt.yticks(sep_psi_mp2_line,sep_psi_mp2_lab)

plt.colorbar()
plt.savefig('PSI2_MP2.png',dpi=400)
plt.show()
plt.imshow(get_masked_matrix(d,mask)[psi_mp2_odering,:][:,psi_mp2_odering],cmap=plt.cm.gist_yarg)
plt.title(str('PSI*2 and MP*2 posterior sample average'))
for i in sep_psi_mp2:
    print(i)
    plt.axvline(i - 0.5, color='blue')
    plt.axhline(i - 0.5, color='blue')
plt.xticks(sep_psi_mp2_line,sep_psi_mp2_lab)
plt.xticks(rotation='vertical')
plt.yticks(sep_psi_mp2_line,sep_psi_mp2_lab)

plt.colorbar()
plt.savefig('PSI2_MP2.png',dpi=400)
plt.show()

In [ ]:
create_part_matr(np.array([0,2,0,0,0,2,2,2]))

In [ ]:
"""m=np.load('hs_inf_mtx.npy')"""

In [ ]:
"""p=np.load('hs_inf_part.npy')
t=[]
for i in p:
    t.append(len(set(i[0])))
"""

In [ ]:
"""p[0][0]"""

In [ ]:
import pandas as pd

In [ ]:
classes

In [ ]:
num_of_day='04'
days=['02','03','04','05','06']

In [ ]:
vector_of_pointests={}
vector_of_entropys={}
for num_of_day in days:
    mx=np.load(os.getcwd() + str("/diff_days/"+'inf_mtx_')+str(num_of_day)+'.npy')
    point_est=sum(mx)/len(mx)
    plt.imshow(point_est,cmap=plt.cm.bwr)
    vector_of_pointests[num_of_day]=point_est
    plt.show()
    import numpy as np
    import pandas as pd
    from matplotlib import pyplot as plt
    from datetime import datetime, timedelta

    df_id=pd.read_csv('ids_with_index.txt',header=None)
    df_id.head()
    ids=pd.DataFrame.to_numpy(df_id)

    df = pd.read_csv('High-School_data_2013.csv', delimiter=' ')
    df.head()


    x=pd.DataFrame.to_numpy(df)
    def lookup(key, dict):
        return dict[key]
    id_dict={}
    for i in ids:
        id_dict[i[1]]=i[0]
    print(id_dict)
    for i in range(len(x)):
        #print(x[i][1])
        #print(id_dict[x[i][1]])
        #id_dict[x[i][1]]
        x[i][2]=id_dict[x[i][2]]
        x[i][1]=id_dict[x[i][1]]
    print(x)
    try:
        classes=classes
    except ValueError:
        classes=set(df['class1'])
    print(classes)
    classes_dict={}
    curr_key=0
    for i in classes:
        classes_dict[i]=curr_key
        curr_key+=1
    classes_dict
    memberships=np.zeros(len(point_est))
    for i in x:
        memberships[i[1]]=classes_dict[i[3]]
        memberships[i[2]]=classes_dict[i[4]]
    print(memberships)

    ordering=np.argsort(memberships)
    classes_reverse = {y: x for x, y in classes_dict.items()}

    def plot_histograms(inferences,parts,savetitle=None,plt_title=None):
        entropy_mtx=np.zeros((len(inferences[0]),len(inferences[0])))
        
        mplus,mneg,mzer=[],[],[]
        for i in inferences:
            mplus.append((i>0).astype(int))
            mneg.append((i<0).astype(int))
            mzer.append((i==0).astype(int))
        
        mplus_agg=sum(mplus)
        mneg_agg=sum(mneg)
        mzer_agg=sum(mzer)
        for n in range(len(mplus_agg)):
            for m in range(n):
                entropy_mtx[n][m]=scipy.stats.entropy([mplus_agg[n][m],mneg_agg[n][m],mzer_agg[n][m]])
                entropy_mtx[m][n]=entropy_mtx[n][m]
        return entropy_mtx
        temp=np.zeros((len(parts[0]),len(parts[0])))
        
        for i in range(len(temp)):
            for j in range(i):
                found=False
                curr=0
                while found==False and curr <len(parts):
                    if parts[curr][i]==parts[curr][j]:
                        found=True
                        temp[i][j]=1
                        temp[j][i]=1
                    curr+=1
        tgthr=[]
        sprtly=[]
        for i in range(len(entropy_mtx)):
            for j in range(i):
                if temp[i][j]==1:
                    tgthr.append(entropy_mtx[i][j])
                else:
                    sprtly.append(entropy_mtx[i][j])
        """
        plt.hist([tgthr, sprtly], color=['Blue', 'Red'], label=['Edge inside a partition', 'Edge between partitions '])
        plt.xlabel('Entropy')
        plt.title(plt_title)
        """
        tgthr=np.transpose(tgthr)
        sprtly=np.transpose(sprtly)
        new_vec1=[]
        new_vec2=[]
        for i in range(len(tgthr)):
            new_vec1.append(tgthr[i])
            new_vec2.append("Inside partition")
        for i in range(len(sprtly)):
            new_vec1.append(sprtly[i])
            new_vec2.append("Between partitions")

        df=pd.DataFrame([new_vec1,new_vec2])
        """
        plt.ylabel('Entropy')
        plt.legend()
        plt.savefig('asdf')
        if savetitle!=None:
            plt.savefig(str(savetitle))
        plt.show()
        """
        #sns.histplot([tgthr,sprtly],kde=True,cumulative=False,cbar=True)
        df=df.transpose()
        df=df.rename(columns={0:'Entropy',1:'Edges'})
        sns.histplot(data=df, x='Entropy',palette=['#154cb0', '#de1212'], hue='Edges',kde=True,multiple="layer")
        plt.title(plt_title)
        plt.savefig(savetitle)
        plt.show()
        return df
    partitions=np.array(np.zeros((len(point_est),1)))
    d=plot_histograms(mx,partitions,'cico.png')
    vector_of_entropys[num_of_day]=d
    separators=[0]
    separators_lab=[classes_reverse[memberships[ordering[0]]]]
    for i in range(len(ordering)-1):
        if memberships[ordering[i]]!=memberships[ordering[i+1]]:
            separators.append(i)
            separators_lab.append(classes_reverse[memberships[ordering[i+1]]])
    print(separators)
    print(classes_reverse)
    print(separators_lab)

    #separators=[]
    for i in classes:
        print(i)
        for j in ordering:
            print(j)


    separarators_plot=np.copy(separators)
    separarators_plot=list(separarators_plot)
    separarators_plot.append(327)
    temp=[]
    for i in range(len(separarators_plot)-1):
        temp.append((separarators_plot[i]+separarators_plot[i+1])/2)
    print(len(temp))
    separarators_plot=temp
    len(separarators_plot)


    plt.imshow(d[ordering,:][:,ordering],cmap=plt.cm.gist_yarg)
    for i in separators:
        print(i)
        plt.axvline(i - 0.5, color='blue')
        plt.axhline(i - 0.5, color='blue')
    plt.xticks(separarators_plot,separators_lab)
    plt.xticks(rotation='vertical')
    plt.yticks(separarators_plot,separators_lab)
    plt.title('Entropy of inference on December '+str(num_of_day))
    plt.colorbar()
    plt.savefig('high_school_entropy_dec'+str(num_of_day)+'.svg',dpi=500)
    plt.show()
    plt.imshow(point_est[ordering,:][:,ordering],cmap=plt.cm.bwr,vmin=-1)
    for i in separators:
        print(i)
        plt.axvline(i - 0.5, color='black')
        plt.axhline(i - 0.5, color='black')
    plt.xticks(separarators_plot,separators_lab)
    plt.xticks(rotation='vertical')
    plt.yticks(separarators_plot,separators_lab)
    plt.title('Sample mean of inference on December '+str(num_of_day))
    plt.colorbar()
    plt.savefig('high_school_sampleavrg_dec'+str(num_of_day)+'.svg',dpi=500)

In [ ]:
obs_12_06=np.load('observation_mtx2013-12-06.npy')
obs_agg=sum(obs_12_06)

In [ ]:
plt.imshow(obs_agg,cmap=plt.cm.Purples,vmax=1)
#plt.colorbar()
plt.savefig('obs_mtx.pdf')

In [ ]:
plt.imshow(obs_agg[ordering,:][:,ordering],cmap=plt.cm.Purples,vmax=1)
for i in separators:
    print(i)
    plt.axvline(i - 0.5, color='blue')
    plt.axhline(i - 0.5, color='blue')
plt.xticks(separarators_plot,separators_lab)
plt.xticks(rotation='vertical')
plt.yticks(separarators_plot,separators_lab)
plt.title('Pairs of nodes that have\n interacted on December '+str('03'))
plt.savefig('interaction_nodes.svg')

In [ ]:

fig, ax = plt.subplots(1, 2,constrained_layout=True,figsize=(8,4))

im1=ax[0].imshow(d[ordering,:][:,ordering],cmap=plt.cm.gist_yarg)
for i in separators:
    print(i)
    ax[0].axvline(i - 0.5, color='blue')
    ax[0].axhline(i - 0.5, color='blue')
ax[0].set_xticks(separarators_plot,separators_lab,rotation='vertical',size=12)
#ax[0].set_xticks()
ax[0].set_yticks(separarators_plot,separators_lab,size=12)
ax[0].set_title('Entropy of inference \n on December '+str(num_of_day))
plt.colorbar(im1,ax=ax[0],fraction=0.05 )
#plt.savefig('high_school_entropy_dec'+str(num_of_day)+'.svg',dpi=500)
#plt.show()
im2=ax[1].imshow(point_est[ordering,:][:,ordering],cmap=plt.cm.bwr,vmin=-1)
for i in separators:
    print(i)
    ax[1].axvline(i - 0.5, color='black')
    ax[1].axhline(i - 0.5, color='black')
ax[1].set_xticks(separarators_plot,separators_lab,rotation='vertical',size=12)
ax[1].set_yticks(separarators_plot,separators_lab,size=12)
ax[1].set_title('Sample mean of inference on \n December '+str(num_of_day))
plt.colorbar(im2,ax=ax[1],fraction=0.05)
plt.savefig('graph_for_paper_dec'+str(num_of_day)+'.svg',dpi=500)

In [ ]:
point_est[ordering,:][:,ordering]

In [ ]:
len(np.resize(point_est,len(point_est)**2))
point_est_save=point_est

In [ ]:
for num_of_day in days:
    mx=np.load(os.getcwd() + str("/diff_days/"+'inf_mtx_')+str(num_of_day)+'.npy')
    point_est=sum(mx)/len(mx)
    #point_est=(point_est>0.5).astype(int)
    plt.imshow(point_est,cmap=plt.cm.bwr,vmin=-1)
    plt.show()
    vector_of_pointests[num_of_day]=point_est

In [ ]:
corr_mtx=np.zeros((5,5))
corr_mtx_entropy=np.zeros((5,5))
x,y=-1,0
for i in vector_of_pointests.keys():
    x+=1
    y=0
    for j in vector_of_pointests.keys():
        corr_mtx_entropy[x][y]=scipy.stats.pearsonr(np.resize(vector_of_entropys[i],len(vector_of_entropys[i])**2),np.resize(vector_of_entropys[j],len(vector_of_entropys[j])**2))[0]
        #print('ENTROPY',scipy.stats.pearsonr(np.resize(vector_of_entropys[i],len(vector_of_entropys[i])**2),np.resize(vector_of_entropys[j],len(vector_of_entropys[j])**2)))

        print(scipy.stats.pearsonr(np.resize(vector_of_pointests[i],len(vector_of_pointests[i])**2),np.resize(vector_of_pointests[j],len(vector_of_pointests[j])**2)))
        corr_mtx[x][y]=scipy.stats.pearsonr(np.resize(vector_of_pointests[i],len(vector_of_pointests[i])**2),np.resize(vector_of_pointests[j],len(vector_of_pointests[j])**2))[0]
        y+=1

In [ ]:
print(corr_mtx)

In [ ]:
print(scipy.stats.pearsonr(np.resize(vector_of_pointests[i],len(vector_of_pointests[i])**2),np.resize(vector_of_pointests[j],len(vector_of_pointests[j])**2))[0])


In [ ]:
friendships=np.genfromtxt('Friendship-network_data_2013.csv', delimiter=' ', dtype=None)


In [ ]:
id_dict

In [ ]:
friendships

In [ ]:
friendships

In [ ]:
for i in range(len(friendships)):
    for j in range(2):
        friendships[i][j]=id_dict[friendships[i][j]]

In [ ]:
ll=[]
for i in friendships:
    l=[]
    for j in vector_of_pointests.keys():
        l.append(vector_of_pointests[j][i[0]][i[1]])
    print(sum(l))
    ll.append(sum(l)/5)

In [ ]:
ll

In [ ]:
nums=1000
indices=[]
for i in range(nums):
    a=np.random.choice(327)
    b=a
    while b==a:
        b=np.random.choice(327)
    indices.append([a,b])
    print(a,b)

In [ ]:
ll_friends=[]
for i in friendships:
    l=[]
    for j in vector_of_pointests.keys():
        l.append(vector_of_pointests[j][i[0]][i[1]])
    print(sum(l))
    ll_friends.append(sum(l)/5)
plt.hist(ll_friends,bins=40)

In [ ]:
ll_random=[]
for i in indices:
    l=[]
    for j in vector_of_pointests.keys():
        l.append(vector_of_pointests[j][i[0]][i[1]])
    print(sum(l))
    ll_random.append(sum(l)/5)
plt.hist(ll_random,bins=40)

In [ ]:
sns.histplot(ll_friends,bins=20,color='salmon',label='Cic')
sns.histplot(ll_random,bins=20,color='dodgerblue')

In [ ]:
plt.hist(ll_friends,color='salmon',alpha=0.7,bins=20,label='Pair of friends')
plt.hist(ll_random,color='dodgerblue',alpha=0.5,bins=20,label='Pair of randomly chosen nodes')
plt.xlabel('Average of sample drawn from the posterior ')
plt.title('Histograms of sample mean between friends random nodes ')
plt.ylabel('Counts')
plt.legend()
plt.savefig('friendship plot.svg')


In [ ]:
plt.hist(ll,bins=40)

# Check reciprocated/ unreciprocated friendships

In [ ]:
def friendships_reciprocated(l):
    recipr=[]
    unrecipr=[]
    recip=False
    for x in l:
        recip=False
        for y in l:
            if x[1]==y[0] and x[0]==y[1]:
                recip=True
                recipr.append((min(x[0],x[1]),max(x[0],x[1])))
        if recip==False:
            unrecipr.append((min(x[0],x[1]),max(x[0],x[1])))
    return recipr,unrecipr
r,ur=friendships_reciprocated(friendships)

def anydup(thelist):
  seen = set()
  for x in thelist:
    if x in seen: return True
    seen.add(x)
  return False
print(anydup(r))

In [ ]:
ll_friends_r=[]
for i in r:
    l_r=[]
    for j in vector_of_pointests.keys():
        l_r.append(vector_of_pointests[j][i[0]][i[1]])
    print(sum(l_r))
    ll_friends_r.append(sum(l_r)/5)
plt.hist(ll_friends_r,bins=40)
#plt.show()
ll_friends_ur=[]
for i in ur:
    l_ur=[]
    for j in vector_of_pointests.keys():
        l_ur.append(vector_of_pointests[j][i[0]][i[1]])
    print(sum(l_ur))
    ll_friends_ur.append(sum(l_ur)/5)
plt.hist(ll_friends_ur,bins=40)
nums=300
indices=[]
for i in range(nums):
    a=np.random.choice(327)
    b=a
    while b==a:
        b=np.random.choice(327)
    indices.append([a,b])
    print(a,b)

ll_random=[]
for i in indices:
    l=[]
    for j in vector_of_pointests.keys():
        l.append(vector_of_pointests[j][i[0]][i[1]])
    print(sum(l))
    ll_random.append(sum(l)/5)
plt.hist(ll_random,bins=40)

In [ ]:
plt.figure(figsize=(8, 4), dpi=80)
plt.hist(ll_friends_r,bins=30,color='salmon',alpha=0.8,label='Pair of reciprocated friends',range=(-0.2,1))
plt.hist(ll_friends_ur,bins=30,color='orchid',edgecolor='orchid',alpha=0.6,label='Pair of unreciprocated friends',range=(-0.2,1))

plt.hist(ll_random,bins=30,color='dodgerblue',alpha=0.4,label='Pair of randomly chosen nodes',range=(-0.2,1))
plt.xlabel('Mean of sample drawn from the posterior ',size=14)
plt.title('Histograms of sample mean between friends and \n randomly selected nodes ',size=14)
plt.ylabel('Counts',size=14)
plt.legend()
plt.savefig('friendship plot.svg')


In [ ]:
plt.hist(ll_friends_r, color='salmon', edgecolor='salmon', alpha=0.8, label='Pair of reciprocated friends')
plt.hist(ll_friends_ur, color='orchid', edgecolor='orchid', alpha=0.6, label='Pair of unreciprocated friends')
plt.hist(ll_random, color='dodgerblue', edgecolor='dodgerblue', alpha=0.4, label='Pair of randomly chosen nodes')

plt.xlabel('Average of sample drawn from the posterior', size=14)
plt.title('Histograms of sample mean between friends and random nodes', size=14)
plt.ylabel('Counts', size=14)
plt.legend(fontsize=12)
plt.rc('legend',fontsize=12) # using a size in points
plt.savefig('friendship_plot.pdf')

plt.show()

In [ ]:
pos=0
neg=0
for i in ll:
    if i>0:
        pos+=1
    else:
        neg+=1
print(pos,neg)

In [ ]:
scipy.stats.entropy([1,1],[1,1])

In [ ]:
l

In [ ]:
for num_of_day in days:
    mx=np.load(os.getcwd() + str("/diff_days/"+'inf_mtx_')+str(num_of_day)+'.npy')
    point_est=sum(mx)/len(mx)
    
    plt.imshow(point_est,cmap=plt.cm.bwr,vmin=-1)
    plt.show()
    vector_of_pointests[num_of_day]=point_est

In [ ]:
def distr_of_elements(inferences,parts):
    entropy_mtx=np.zeros((len(inferences[0]),len(inferences[0])))
    entropy_mtx=list(entropy_mtx)

    for i in range(len(entropy_mtx)):
        entropy_mtx[i]=list(entropy_mtx)
    mplus,mneg,mzer=[],[],[]
    
    for i in inferences:
        mplus.append((i>0).astype(int))
        mneg.append((i<0).astype(int))
        mzer.append((i==0).astype(int))
    
    mplus_agg=sum(mplus)
    mneg_agg=sum(mneg)
    mzer_agg=sum(mzer)
    return mplus_agg,mneg_agg,mzer_agg
    for n in range(len(mplus_agg)):
        print(n)
        for m in range(len(mplus_agg)):
            entropy_mtx[n][m]=[[mplus_agg[n][m],mneg_agg[n][m],mzer_agg[n][m]]]
            entropy_mtx[m][n]=entropy_mtx[n][m]
    return entropy_mtx

In [ ]:
dist=distr_of_elements(mx,None)
print(dist)

In [ ]:
vector_of_infs={}
for num_of_day in days:
    mx=np.load(os.getcwd() + str("/diff_days/"+'inf_mtx_')+str(num_of_day)+'.npy')
    
    
    vector_of_infs[num_of_day]=distr_of_elements(mx,None)

In [ ]:
print(vector_of_infs)

In [ ]:
entropys={}
for x in days:
    for i in range(len(vector_of_entropys[x])):
            for j in range(len(vector_of_entropys[x])):
                if vector_of_entropys[x][i][j]<0.8 and i !=j:
                    try:
                         entropys[x].append([i,j])
                    except:
                         entropys[x]=[[i,j]]
       

In [ ]:
def intersection(l1,l2):
    t=[]
    for x in l1:
        for y in l2:
            if set(x)==set(y) and x not in t:
                t.append(x)
    return t
intersection(entropys['02'],entropys['05'])

In [ ]:
f=np.zeros((5,5))
for x in range(len(days)):
    for y in range(len(days)):
        f[x][y]=len(intersection(entropys[days[x]],entropys[days[y]]))
        f[y][x]=f[x][y]
print(f)

In [ ]:
print(f)

In [ ]:
scipy.stats.entropy([1,1,1],[2,3,1])

In [ ]:
def tot_var_dist(p,q):
    dist=0
    sump=sum(p)
    sumq=sum(q)
    for x in range(len(p)):
        dist+=np.abs(p[x]/sump-q[x]/sumq)
    return dist/2
print(tot_var_dist([0,1],[1,0]))

In [ ]:
print(entropys['02'])
print(entropys['03'])
print(entropys['04'])
print(entropys['05'])
print(entropys['06'])

In [ ]:
cl=-2**30
for x in days:
    
    for y in days:
        print(x,y)
        t=[]
        if x!=y and int(x)<int(y):
            t=intersection(entropys[x],entropys[y])
        dists=[]
        for t1 in t:
            if cl<vector_of_infs[x][2][t1[0]][t1[1]]:
                cl=vector_of_infs[x][2][t1[0]][t1[1]]
            tempx=[vector_of_infs[x][0][t1[0]][t1[1]],vector_of_infs[x][1][t1[0]][t1[1]],vector_of_infs[x][2][t1[0]][t1[1]]]
            tempy=[vector_of_infs[y][0][t1[0]][t1[1]],vector_of_infs[y][1][t1[0]][t1[1]],vector_of_infs[y][2][t1[0]][t1[1]]]
            
            dists.append(tot_var_dist(tempx,tempy))
        if dists!=[]:
            print(sum(dists)/len(dists))

In [ ]:
cl=-2**30
mtr=np.ones((5,5))
for x in range(len(days)):
    
    for y in range(len(days)):
        print(days[x],days[y])
        t=[]
        if days[x]!=days[y] and int(days[x])<int(days[y]):
            t=intersection(entropys[days[x]],entropys[days[y]])
        dists=[]
        for t1 in t:
            if cl<vector_of_infs[days[x]][2][t1[0]][t1[1]]:
                cl=vector_of_infs[days[x]][2][t1[0]][t1[1]]
            tempx=[vector_of_infs[days[x]][0][t1[0]][t1[1]],vector_of_infs[days[x]][1][t1[0]][t1[1]],vector_of_infs[days[x]][2][t1[0]][t1[1]]]
            tempy=[vector_of_infs[days[y]][0][t1[0]][t1[1]],vector_of_infs[days[y]][1][t1[0]][t1[1]],vector_of_infs[days[y]][2][t1[0]][t1[1]]]
            
            dists.append(tot_var_dist(tempx,tempy))
        if dists!=[]:
            #print(sum(dists)/len(dists))
            mtr[x][y]=sum(dists)/len(dists)
            mtr[y][x]=mtr[x][y]

In [ ]:
print(mtr)

In [ ]:
cl

In [ ]:
obs_12_06=np.load('observation_mtx2013-12-06.npy')
obs_agg=sum(obs_12_06)

In [ ]:
plt.imshow(obs_agg[ordering,:][:,ordering],cmap=plt.cm.Purples,vmax=1)
for i in separators:
    print(i)
    plt.axvline(i - 0.5, color='blue')
    plt.axhline(i - 0.5, color='blue')
plt.xticks(separarators_plot,separators_lab)
plt.xticks(rotation='vertical')
plt.yticks(separarators_plot,separators_lab)
plt.title('Pairs of nodes that have\n interacted on December '+str('03'))
plt.savefig('interaction_nodes.svg')

In [ ]:
fig, ax = plt.subplots(1, 3,constrained_layout=True,figsize=(12,4))

im1=ax[2].imshow(d[ordering,:][:,ordering],cmap=plt.cm.gist_yarg)
for i in separators:
    print(i)
    ax[2].axvline(i - 0.5, color='blue')
    ax[2].axhline(i - 0.5, color='blue')
ax[2].set_xticks(separarators_plot,separators_lab,rotation='vertical',size=12)
#ax[0].set_xticks()
ax[2].set_yticks(separarators_plot,separators_lab,size=12)
ax[2].set_title('Entropy of inference \n on December '+str(num_of_day))
plt.colorbar(im1,ax=ax[2],fraction=0.05 )
#plt.savefig('high_school_entropy_dec'+str(num_of_day)+'.svg',dpi=500)
#plt.show()
im2=ax[1].imshow(point_est[ordering,:][:,ordering],cmap=plt.cm.bwr,vmin=-1)
for i in separators:
    print(i)
    ax[1].axvline(i - 0.5, color='black')
    ax[1].axhline(i - 0.5, color='black')
ax[1].set_xticks(separarators_plot,separators_lab,rotation='vertical',size=12)
ax[1].set_yticks(separarators_plot,separators_lab,size=12)
ax[1].set_title('Sample mean of inference on \n December '+str(num_of_day))
plt.colorbar(im2,ax=ax[1],fraction=0.05)


im3=ax[0].imshow(obs_agg[ordering,:][:,ordering],cmap=plt.cm.Purples,vmax=1)
for i in separators:
    print(i)
    ax[0].axvline(i - 0.5, color='blue')
    ax[0].axhline(i - 0.5, color='blue')
ax[0].set_xticks(separarators_plot,separators_lab,rotation='vertical',size=12)
ax[0].set_yticks(separarators_plot,separators_lab,size=12)
#ax[2].set_title('Sample mean of inference on \n December '+str(num_of_day))
ax[0].set_title('Pairs of nodes that have\n interacted on December '+str('06'))
#plt.show()
plt.savefig('graph_for_paper_dec'+str(num_of_day)+'.svg',dpi=500)